# Task 1: Structured LLM Prompt Pipeline & JSON Extractor

**Project:** Hirelyzer, structured JSON extraction from resumes
**Language and stack:** Python 3.12, Pydantic, OpenAI-compatible SDK, OpenRouter (`openai/gpt-oss-20b`), pytest, ruff, uv

This notebook walks through my Task 1 project from top to bottom. It does not copy the code. It imports the `hirelyzer` package, shows the source of each module, and runs the real pipeline, so what you read here is the same code the tests check.

## Contents

1. Objective and key concepts
2. Setup
3. Pipeline architecture
4. Source code tour (config, errors, readers, vision, prompts, llm, parser, schema, extractor, telemetry, evaluate, cli)
5. Schema models
6. Prompt template and few-shot files
7. Live run: extract one resume
8. Error handling: a non-resume and a missing file
9. Telemetry
10. Evaluation results
11. Tests
12. Design decisions, limitations, skills

## 1. Objective and key concepts

**Objective:** read unstructured text (a resume as PDF, DOCX, TXT, image or raw text), ask an LLM to extract its content, and return JSON that is checked against a strict schema. Output the model gets wrong is sent back for repair. A document that still fails is reported, never silently dropped.

**Key concepts**

- Prompt engineering: parameterized template, few-shot examples, schema included in the prompt
- Structured LLM output: JSON mode at `temperature=0.0`
- JSON extraction and parsing: code fences and surrounding prose are stripped
- Pydantic schema validation: strict models, unknown fields rejected, date and contact patterns checked
- Output repair loop: validation errors go back to the model, up to 2 rounds
- API error handling: retry with exponential backoff and jitter, only for transient errors
- Input guards: missing, empty, oversized, corrupt and unsupported files
- Evaluation against ground truth, plus per-request telemetry (latency, tokens, retries)

## 2. Setup

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    nearby = [here, *here.parents] + [p for p in here.iterdir() if p.is_dir()]
    for folder in nearby:
        if (folder / "hirelyzer" / "cli.py").exists():
            return folder
    raise FileNotFoundError("hirelyzer package not found. Open this notebook from the Hirelyzer project folder.")


PROJECT = find_project_root()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("Project folder:", PROJECT.name)
print("Python:", sys.version.split()[0])

Project folder: Task1_Hirelyzer
Python: 3.12.12


In [2]:
# Check that an API key is configured, without printing any secret.
env_file = PROJECT / ".env"
print(".env file found:", env_file.exists())
if env_file.exists():
    names = set()
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            names.add(line.split("=", 1)[0].strip())
    print("API key variable present:", bool(names & {"OPENROUTER_API_KEY", "OPENAI_API_KEY"}))

.env file found: False


## 3. Pipeline architecture

```
input: .pdf / .docx / .txt / raw text            .jpg / .png
        |                                            |
   readers.py (guards, text extraction)       vision.py (guards, data URL)
        |                                            |
        +------------> prompts.py <------------------+   template + few-shot + schema
                          |
                    llm.py: call with retry/backoff, telemetry (temperature 0.0, JSON mode)
                          |
                    parser.py: fences and prose stripped -> JSON
                          |
                    schema.py: Pydantic validation
                          |  invalid? send the error back to the model (max 2 repair rounds)
                          v
                    ExtractionResult { status, data, errors, attempts, tokens, latency }
```

Clients are passed in (dependency injection) and nothing runs at import time, so every stage can be tested with fakes.

## 4. Source code tour

The helper below prints the source of a module straight from the package, so this notebook always matches the real code.

In [3]:
import importlib
import inspect


def show_source(module_name: str) -> None:
    module = importlib.import_module(module_name)
    source = inspect.getsource(module)
    print(f"# {module_name}  ({len(source.splitlines())} lines)\n")
    print(source)

### 4.1 Configuration (`config.py`)

Settings come from environment variables or `.env` and are validated once at startup. A missing key stops the program with a readable message.

In [4]:
show_source("hirelyzer.config")

# hirelyzer.config  (105 lines)

"""Runtime settings for Hirelyzer, read from environment variables.

Nothing here runs at import time. Call load_settings() once at startup.
"""

import os
from collections.abc import Callable, Mapping
from dataclasses import dataclass, field

from hirelyzer.errors import ConfigError

# Defaults live here as named constants, not buried inside functions.
DEFAULT_BASE_URL = "https://openrouter.ai/api/v1"
DEFAULT_TEXT_MODEL = "openai/gpt-oss-20b"
DEFAULT_VISION_MODEL = "qwen/qwen3.7-flash"
DEFAULT_TEMPERATURE = 0.0
DEFAULT_MAX_RETRIES = 3
DEFAULT_MAX_REPAIR_ATTEMPTS = 2
DEFAULT_MAX_FILE_MB = 10
DEFAULT_MAX_TEXT_CHARS = 60_000

# The first variable that is set wins, so OpenRouter or OpenAI keys both work.
API_KEY_VARS = ("OPENROUTER_API_KEY", "OPENAI_API_KEY")


@dataclass(frozen=True)
class Settings:
    """All runtime settings, validated once at startup and never changed."""

    api_key: str = field(repr=False)  # repr=False keeps the key out of logs
   

### 4.2 Typed errors (`errors.py`)

Every failure the program expects has its own exception type, so the CLI can turn it into a clear one-line message.

In [5]:
show_source("hirelyzer.errors")

# hirelyzer.errors  (33 lines)

"""Exception hierarchy for Hirelyzer.

Every error the pipeline raises on purpose inherits from HirelyzerError,
so callers can catch one base class and still tell the cases apart.
"""


class HirelyzerError(Exception):
    """Base class for all Hirelyzer errors."""


class ConfigError(HirelyzerError):
    """Required configuration is missing or invalid (raised at startup)."""


class UnsupportedFileError(HirelyzerError):
    """The input file type is not supported."""


class EmptyDocumentError(HirelyzerError):
    """The file, or the text extracted from it, contains nothing to analyse."""


class InputTooLargeError(HirelyzerError):
    """The input file exceeds the configured size limit."""


class LLMCallError(HirelyzerError):
    """The LLM request failed for good: a non-transient error, or retries ran out."""


class OutputValidationError(HirelyzerError):
    """The model output could not be parsed or validated, even after repair attempts."""



### 4.3 Document readers (`readers.py`)

PDF, DOCX and TXT to text, with guards for missing, empty, oversized, corrupt and unsupported files.

In [6]:
show_source("hirelyzer.readers")

# hirelyzer.readers  (110 lines)

"""Read a document (PDF, DOCX, TXT) or raw text into clean text, with input guards."""

import logging
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

from hirelyzer.errors import EmptyDocumentError, InputTooLargeError, UnsupportedFileError
from hirelyzer.vision import IMAGE_EXTENSIONS, image_to_data_url

logger = logging.getLogger(__name__)


def _read_txt(path: Path) -> str:
    """Read a plain text file; undecodable bytes become replacement characters."""
    return path.read_text(encoding="utf-8", errors="replace")


def _read_pdf(path: Path) -> str:
    """Extract markdown text from a PDF with pymupdf4llm."""
    # Imported here so the text-only path works without loading the PDF stack.
    import pymupdf4llm

    try:
        return pymupdf4llm.to_markdown(str(path))
    except Exception as exc:  # third-party parsers raise many different error types
        raise UnsupportedFileError(f"Cannot read

### 4.4 Image input (`vision.py`)

Images are checked and converted to data URLs for the vision model.

In [7]:
show_source("hirelyzer.vision")

# hirelyzer.vision  (35 lines)

"""Turn an image file into a data URL for the vision model, with input guards."""

import base64
from pathlib import Path

from hirelyzer.errors import EmptyDocumentError, InputTooLargeError, UnsupportedFileError

# File extension -> MIME type used in the data URL.
IMAGE_TYPES = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}
IMAGE_EXTENSIONS = frozenset(IMAGE_TYPES)


def image_to_data_url(path: Path, max_file_bytes: int) -> str:
    """Read an image file and return it as a base64 data URL, or raise a typed error."""
    mime = IMAGE_TYPES.get(path.suffix.lower())
    if mime is None:
        supported = ", ".join(sorted(IMAGE_TYPES))
        raise UnsupportedFileError(
            f"Unsupported image type {path.suffix!r} for {path.name}. Supported: {supported}"
        )
    size = path.stat().st_size
    if size == 0:
        raise EmptyDocumentError(f"{path.name} is empty.")
    if size > max_file_bytes:
        raise InputTooLarge

### 4.5 Prompt building (`prompts.py`)

A parameterized template plus few-shot examples and the schema. The template files live on disk and are validated at startup.

In [8]:
show_source("hirelyzer.prompts")

# hirelyzer.prompts  (147 lines)

"""Build the LLM messages: a parameterized system prompt plus the document text.
The template and the few-shot examples live in the prompts/ folder, not in code.
They are loaded and validated once at startup, so a broken prompt file fails fast.
"""

import json
import logging
import re
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Self

from pydantic import ValidationError

from hirelyzer.errors import ConfigError
from hirelyzer.schema import Resume

logger = logging.getLogger(__name__)

PROMPTS_DIR = Path(__file__).resolve().parent.parent / "prompts"
TEMPLATE_FILE = "system.txt"
EXAMPLES_FILE = "examples.json"
PLACEHOLDERS = ("schema", "examples", "extra_rules")
_PLACEHOLDER_RE = re.compile(r"\{(" + "|".join(PLACEHOLDERS) + r")\}")


@dataclass(frozen=True)
class FewShotExample:
    """One worked example: a document and the JSON the model should return for it."""

    d

### 4.6 LLM client (`llm.py`)

OpenAI-compatible client. Retries only transient errors (429, timeouts, connection errors, 5xx) with exponential backoff and jitter. Auth errors and bad requests are never retried.

In [9]:
show_source("hirelyzer.llm")

# hirelyzer.llm  (164 lines)

"""Thin wrapper around the OpenAI-compatible client: retries, backoff and telemetry hooks."""

import logging
import random
import time
from collections.abc import Callable
from dataclasses import dataclass

import openai

from hirelyzer.config import Settings
from hirelyzer.errors import LLMCallError
from hirelyzer.telemetry import RequestRecord, TelemetryRecorder

logger = logging.getLogger(__name__)

REQUEST_TIMEOUT_S = 60.0


def backoff_delay(attempt: int, base: float = 1.0, cap: float = 30.0) -> float:
    """Seconds to wait before retry number `attempt` (1 = first retry), with full jitter."""
    ceiling = min(cap, base * 2 ** (attempt - 1))
    return random.uniform(0, ceiling)


def is_transient(exc: Exception) -> bool:
    """Return True if the error is worth retrying."""
    if isinstance(exc, (openai.APITimeoutError, openai.APIConnectionError)):
        return True
    if isinstance(exc, openai.APIStatusError):
        if exc.status_code == 429

### 4.7 Output parser (`parser.py`)

Turns raw model text into a JSON object. It handles code fences and surrounding prose. Truncated JSON fails loudly.

In [10]:
show_source("hirelyzer.parser")

# hirelyzer.parser  (36 lines)

"""Turn raw model output into a JSON object, tolerating code fences and extra prose."""

import json
import re
from typing import Any

from hirelyzer.errors import OutputValidationError

# Matches ```json ... ``` or ``` ... ``` and captures what is inside the fence.
_FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)


def strip_code_fence(text: str) -> str:
    """Return the text inside the first markdown code fence, or the text unchanged."""
    match = _FENCE_RE.search(text)
    return match.group(1).strip() if match else text.strip()


def parse_model_output(raw: str) -> dict[str, Any]:
    """Parse the model's reply into a dict; raise OutputValidationError if it is not valid JSON."""
    text = strip_code_fence(raw)

    # Skip any prose before the JSON. An empty or prose-only reply has no "{" at all.
    start = text.find("{")
    if start == -1:
        raise OutputValidationError("No JSON object found in the model output

### 4.8 Schema (`schema.py`)

Pydantic models and validators: unknown fields rejected, email and phone patterns, `YYYY-MM` dates, end date not before start date.

In [11]:
show_source("hirelyzer.schema")

# hirelyzer.schema  (193 lines)

"""Pydantic models describing the JSON that Hirelyzer extracts from a resume.
The Resume schema is also sent to the LLM inside the system prompt, so the
Field descriptions below double as instructions for the model.
"""

from typing import Annotated, Any, Self

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    StringConstraints,
    field_validator,
    model_validator,
)

# Patterns avoid lookaheads: pydantic's default regex engine does not support them.
EMAIL_PATTERN = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
PHONE_PATTERN = r"^[0-9+()\-\s]+$"
YEAR_MONTH_PATTERN = r"^[0-9]{4}-(0[1-9]|1[0-2])$"
YEAR_PATTERN = r"^[0-9]{4}$"
MIN_PHONE_DIGITS = 7
MAX_PHONE_DIGITS = 15

YearMonth = Annotated[str, StringConstraints(pattern=YEAR_MONTH_PATTERN)]
Year = Annotated[str, StringConstraints(pattern=YEAR_PATTERN)]


def _blank_to_none(value: Any) -> Any:
    """Treat an empty or whitespace-only string as a missing value."""
    if isinstance(value, str) and 

### 4.9 Extractor (`extractor.py`)

Runs the stages, the repair loop and builds the result envelope. A failed extraction is a result, not an exception, so batches keep going.

In [12]:
show_source("hirelyzer.extractor")

# hirelyzer.extractor  (111 lines)

"""Orchestrate one extraction: build prompt, call the model, parse, validate, repair."""

import logging
import time
from dataclasses import dataclass, field
from typing import Any, Literal

from pydantic import ValidationError

from hirelyzer.errors import LLMCallError, OutputValidationError
from hirelyzer.llm import LLMClient
from hirelyzer.parser import parse_model_output
from hirelyzer.prompts import PromptBuilder
from hirelyzer.schema import Resume

logger = logging.getLogger(__name__)

MAX_ERROR_CHARS = 1500  # keep the error text sent back to the model short

REPAIR_TEMPLATE = (
    "Your previous reply was rejected:\n{error}\n\n"
    "Return the corrected JSON object only, following the same schema and rules."
)


@dataclass
class ExtractionResult:
    """Outcome of one extraction. A failed extraction is a result, not an exception."""

    source: str
    status: Literal["success", "failed"] = "failed"
    data: Resume | None = None
    error

### 4.10 Telemetry (`telemetry.py`)

Writes one JSON line per LLM request with latency, tokens and retries.

In [13]:
show_source("hirelyzer.telemetry")

# hirelyzer.telemetry  (80 lines)

"""Per-request telemetry: latency, token usage and retries, logged as JSON lines."""

import json
import logging
from dataclasses import asdict, dataclass, field
from datetime import UTC, datetime
from pathlib import Path

DEFAULT_TELEMETRY_LOG = "logs/telemetry.jsonl"
TELEMETRY_LOGGER_NAME = "hirelyzer.telemetry"

telemetry_logger = logging.getLogger(TELEMETRY_LOGGER_NAME)


@dataclass(frozen=True)
class RequestRecord:
    """Everything worth knowing about one LLM request."""

    source: str  # file name or "text-input"
    model: str
    pipeline: str  # "text" now, "vision" later
    latency_ms: float  # total time including any retry waits
    prompt_tokens: int
    completion_tokens: int
    retries: int  # attempts beyond the first
    success: bool
    error: str = ""
    timestamp: str = field(default_factory=lambda: datetime.now(UTC).isoformat(timespec="seconds"))


def configure_telemetry_log(path: Path) -> logging.Handler:
    """Send tele

### 4.11 Evaluation (`evaluate.py`)

Scores output against ground truth and keeps a history of runs.

In [14]:
show_source("hirelyzer.evaluate")

# hirelyzer.evaluate  (630 lines)

"""Score extraction output against ground truth and track the results from run to run.

Run these from the 01_Hirelyzer folder:
    python -m hirelyzer.evaluate                      # run the model on the synthetic resumes
    python -m hirelyzer.evaluate --outputs some/dir   # score saved outputs, no API calls
    python -m hirelyzer.evaluate --unlabeled my/dir   # resumes without ground truth
"""

import argparse
import csv
import hashlib
import json
import logging
import sys
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field
from datetime import datetime
from functools import partial
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel

from hirelyzer.cli import build_extractor, extract_one, write_result
from hirelyzer.config import Settings, load_settings
from hirelyzer.errors import HirelyzerError
from hirelyzer.extractor import Extract

### 4.12 Command line (`cli.py`)

Argument parsing, batch isolation, run summary and exit codes.

In [15]:
show_source("hirelyzer.cli")

# hirelyzer.cli  (190 lines)

"""Command-line entry point: read documents, extract resumes, write JSON, report a summary."""

import argparse
import logging
import sys
from collections.abc import Callable, Sequence
from functools import partial
from pathlib import Path

from dotenv import load_dotenv

from hirelyzer.config import Settings, load_settings
from hirelyzer.errors import ConfigError, HirelyzerError
from hirelyzer.extractor import ExtractionResult, ResumeExtractor
from hirelyzer.llm import LLMClient
from hirelyzer.prompts import PromptBuilder
from hirelyzer.readers import SUPPORTED_EXTENSIONS, LoadedDocument, load_document, prepare_text
from hirelyzer.telemetry import DEFAULT_TELEMETRY_LOG, TelemetryRecorder, configure_telemetry_log

logger = logging.getLogger(__name__)
NOISY_LOGGERS = ("httpx", "httpx2", "httpcore")  # libraries that log every HTTP request

EXIT_OK = 0
EXIT_FAILED = 1  # at least one document failed
EXIT_USAGE = 2  # bad configuration or nothing to process
D

## 5. Schema models

Listing the Pydantic models and their fields directly from the code.

In [16]:
from pydantic import BaseModel
import hirelyzer.schema as schema

for name, obj in vars(schema).items():
    if inspect.isclass(obj) and issubclass(obj, BaseModel) and obj.__module__ == schema.__name__:
        fields = getattr(obj, "model_fields", None) or getattr(obj, "__fields__", {})
        print(f"{name}: {', '.join(fields)}")

Strict: 
Experience: company, job_title, job_location, start_date, end_date, is_current, highlights
Education: institution, degree, field_of_study, start_year, end_year, grade
Project: project_title, project_description, skills_used
Resume: is_resume, language, name, headline, email, phone, location, skills, experience, education, certifications, projects, languages


/tmp/ipykernel_17307/3375659851.py:6: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  fields = getattr(obj, "model_fields", None) or getattr(obj, "__fields__", {})


Rules enforced by the schema: unknown fields are rejected; email and phone patterns; dates as `YYYY-MM` and years as `YYYY`; end not before start; a current job has no end date; blanks become `null`; lists are stripped and de-duplicated. Everything except `is_resume` is optional, so a non-resume validates as `is_resume=false` with empty fields.

## 6. Prompt template and few-shot files

The prompt text lives in files, not inside Python strings, and is validated when the program starts.

In [17]:
prompt_files = sorted(
    p for p in PROJECT.rglob("*")
    if p.is_file() and "prompts" in p.parts and ".venv" not in p.parts
    and p.suffix in {".txt", ".md", ".json", ".jinja", ".yaml", ".yml"}
)
for path in prompt_files:
    text = path.read_text(encoding="utf-8")
    print("=" * 70)
    print(path.relative_to(PROJECT), f"({len(text.splitlines())} lines)")
    print("=" * 70)
    print("\n".join(text.splitlines()[:40]))
if not prompt_files:
    print("No prompt files found under a folder named 'prompts'.")

prompts/examples.json (63 lines)
[
    {
      "document": "Jordan Avery\nBackend Developer\njordan.avery@example.com | +1 555 010 0199 | Springfield\n\nSKILLS\nPython, SQL, Docker, REST APIs\n\nEXPERIENCE\nNorthwind Labs - Backend Developer (Springfield) | Mar 2022 - Present\n- Built internal reporting APIs used by 4 teams\n- Cut report generation time from 40s to 6s\n\nBrightside Apps - Junior Developer (Remote) | Jun 2019 - Feb 2022\n- Maintained a Django web application\n\nEDUCATION\nSpringfield Technical University - B.Sc. Computer Science, 2015 - 2019, Grade: 3.6/4.0\n\nCERTIFICATIONS\nCloud Practitioner Essentials\n\nPROJECTS\nTask Tracker - A small command-line task manager (Python, SQLite)\n\nLANGUAGES\nEnglish, Spanish",
      "output": {
        "is_resume": true,
        "language": "English",
        "name": "Jordan Avery",
        "headline": "Backend Developer",
        "email": "jordan.avery@example.com",
        "phone": "+1 555 010 0199",
        "location": "Springfi

## 7. Live run: extract one resume

This calls the real command line tool on a synthetic PDF resume. It needs a valid API key in `.env` and makes one API call.

In [19]:
def run_cli(*args: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, "-m", "hirelyzer.cli", *args],
        capture_output=True, text=True, cwd=PROJECT,
    )


SAMPLE = PROJECT / "data" / "synthetic" / "resumes" / "01_rohan_deshmukh_classic_single_column.pdf"
result = run_cli(str(SAMPLE))
print("exit code:", result.returncode)
print(result.stdout[-2500:])
print(result.stderr[-1500:])

exit code: 0
rapidocr_api using backend: rapidocr

=== Document parser messages ===
Using RapidOCR for OCR processing.
Processed 1 document(s): 1 succeeded, 0 failed
Average latency: 21455 ms | Total tokens: 5091
LLM requests: 1 (1 ok, 0 failed) | retries: 0
Avg request latency: 21452 ms | Tokens: 3493 prompt + 1598 completion

2026-11-28 15:09:33,281 INFO __main__: Wrote output/01_rohan_deshmukh_classic_single_column.pdf.json



In [7]:
outputs = sorted((PROJECT / "output").glob("*.json"), key=lambda p: p.stat().st_mtime)
if outputs:
    latest = outputs[-1]
    data = json.loads(latest.read_text(encoding="utf-8"))
    payload = data.get("data", data) if isinstance(data, dict) else data
    print("Output file:", latest.name)
    print(json.dumps(payload, indent=2, ensure_ascii=False)[:3500])
else:
    print("No JSON file in output/ yet. Check the run above for errors.")

Output file: 01_rohan_deshmukh_classic_single_column.pdf.json
{
  "is_resume": true,
  "language": "English",
  "name": "Rohan Deshmukh",
  "headline": "Senior Software Engineer",
  "email": "rohan.deshmukh@example.com",
  "phone": "+91 90000 31245",
  "location": "Pune, India",
  "skills": [
    "Python",
    "Java",
    "Django",
    "FastAPI",
    "PostgreSQL",
    "Redis",
    "Docker",
    "Kubernetes",
    "AWS",
    "CI/CD",
    "REST APIs",
    "Git"
  ],
  "experience": [
    {
      "company": "Brightpath Systems",
      "job_title": "Senior Software Engineer",
      "job_location": "Pune",
      "start_date": "2023-04",
      "end_date": null,
      "is_current": true,
      "highlights": [
        "Redesigned the order-processing service, cutting average API latency from 480 ms to 190",
        "Led migration of 14 microservices to Kubernetes, reducing infrastructure cost by 22%",
        "Mentored 4 junior engineers and introduced code-review guidelines adopted by 3 teams"

## 8. Error handling

Two cases that should not crash the pipeline: text that is not a resume, and a file that does not exist.

In [20]:
not_a_resume = (
    "INVOICE 4821. Bill to: Acme Traders. Item: 40 steel brackets, 12.50 each. "
    "Total due: 500.00. Payment within 30 days."
)
result = run_cli("--text", not_a_resume)
print("exit code:", result.returncode)
print(result.stdout[-1500:])
print(result.stderr[-800:])

exit code: 0
Processed 1 document(s): 1 succeeded, 0 failed
Average latency: 1445 ms | Total tokens: 3166
LLM requests: 1 (1 ok, 0 failed) | retries: 0
Avg request latency: 1444 ms | Tokens: 3116 prompt + 50 completion

2026-11-28 15:09:43,145 INFO __main__: Wrote output/text-input.json



In [21]:
result = run_cli("this_file_does_not_exist.pdf")
print("exit code:", result.returncode)
print(result.stdout[-800:])
print(result.stderr[-800:])

exit code: 1
Processed 1 document(s): 0 succeeded, 1 failed
Average latency: 0 ms | Total tokens: 0
  FAILED this_file_does_not_exist.pdf: File not found: this_file_does_not_exist.pdf
LLM requests: 0




Exit codes: `0` all documents succeeded, `1` at least one document failed, `2` configuration or usage problem. A bad file never stops a batch.

## 9. Telemetry

In [22]:
log = PROJECT / "logs" / "telemetry.jsonl"
rows = []
if log.exists():
    rows = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()]

if rows:
    ok = sum(1 for r in rows if r.get("success"))
    prompt = sum(r.get("prompt_tokens", 0) for r in rows)
    completion = sum(r.get("completion_tokens", 0) for r in rows)
    latencies = [r["latency_ms"] for r in rows if "latency_ms" in r]
    print(f"Requests logged: {len(rows)} ({ok} successful)")
    print(f"Tokens: {prompt:,} prompt + {completion:,} completion")
    if latencies:
        print(f"Average latency: {sum(latencies) / len(latencies) / 1000:.1f} s")
    print("Last entry:")
    print(json.dumps(rows[-1], indent=2))
else:
    print("No telemetry log yet. Run the extraction above first.")

Requests logged: 6 (6 successful)
Tokens: 20,181 prompt + 6,972 completion
Average latency: 13.5 s
Last entry:
{
  "source": "text-input",
  "model": "openai/gpt-oss-20b",
  "pipeline": "text",
  "latency_ms": 1444.0,
  "prompt_tokens": 3116,
  "completion_tokens": 50,
  "retries": 0,
  "success": true,
  "error": "",
  "timestamp": "2026-11-28T09:39:43+00:00"
}


## 10. Evaluation results

`hirelyzer.evaluate` scores output against ground truth. Flat fields are exact matches after normalization. List fields use set precision, recall and F1. Jobs, schools and projects are paired and compared field by field. Every run is saved with the model name, prompt hash, tokens and latency.

**Baseline:** 15 synthetic documents (8 PDF, 3 images, 4 DOCX, one of them an invoice). Model `openai/gpt-oss-20b`, images with `qwen/qwen3.7-flash`. 0 failed documents.

| Metric | Score |
|---|---|
| Overall (mean) | 0.972 |
| is_resume, name, email, phone, location, language | 1.000 |
| Headline | 0.929 |
| Skills precision / recall / F1 | 1.000 / 0.950 / 0.962 |
| Experience F1 | 0.976 |
| Experience highlights F1 | 0.975 |
| Education F1 | 1.000 |
| Certifications F1 | 1.000 |
| Languages F1 | 0.857 |
| Projects F1 | 1.000 |

Tokens: 57,873 prompt + 31,357 completion. Average latency about 23.3 s per document, and it varies between runs.

These numbers are an upper bound. The documents are synthetic and the ground truth was written for them, so real resumes will score lower.

In [23]:
import csv

history = PROJECT / "evaluation" / "runs" / "history.csv"
if history.exists():
    with history.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.reader(handle))
    for row in rows[:1] + rows[-5:]:
        print(" | ".join(row))
else:
    print("No evaluation history file found at evaluation/runs/history.csv")

run_id | mode | model | prompt_hash | documents | failed | overall | overall_processed | is_resume | language | name | headline | email | phone | location | skills_precision | skills_recall | skills_f1 | certifications_f1 | languages_f1 | experience_f1 | experience_fields | experience_highlights_f1 | education_f1 | education_fields | projects_f1 | projects_skills_used_f1
run_id | mode | model | prompt_hash | documents | failed | overall | overall_processed | is_resume | language | name | headline | email | phone | location | skills_precision | skills_recall | skills_f1 | certifications_f1 | languages_f1 | experience_f1 | experience_fields | experience_highlights_f1 | education_f1 | education_fields | projects_f1 | projects_skills_used_f1
2026-11-12T12-15-16 | live | openai/gpt-oss-20b | 52194edd | 15 | 0 | 0.9716 | 0.9716 | 1.0 | 1.0 | 1.0 | 0.9286 | 1.0 | 1.0 | 1.0 | 1.0 | 0.95 | 0.9615 | 1.0 | 0.8571 | 0.9762 | 1.0 | 0.9754 | 1.0 | 0.9554 | 1.0 | 0.8571


## 11. Tests

In [24]:
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True, text=True, cwd=PROJECT,
)
print(tests.stdout[-1500:])
print(tests.stderr[-500:])

................. [ 37%]
........................................................................ [ 74%]
.................................................                        [100%]
193 passed in 1.52s




The tests use fakes for the API client, so they need no network and no API key. They cover the parser, schema, retry and backoff, the repair loop, readers, the CLI and the evaluation code.

## 12. Design decisions, limitations, skills

**Design decisions**

- **OpenAI SDK, not LangChain.** The Task 1 brief names the OpenAI and Gemini SDKs.
- **JSON mode, the schema in the prompt, then Pydantic validation.** The vision model has no JSON-schema enforcement, so I do not trust the API to enforce the schema.
- **`temperature=0.0`** makes output more repeatable, but it does not prevent hallucination. I saw run-to-run variation.
- **Retries are written in visible code.** The SDK's own retries are turned off, so every retry is logged and tested.
- **A failed extraction is a result, not an exception**, so a batch continues and the summary lists what failed.

**Limitations**

- Accuracy is measured on synthetic data only.
- The image path sends the whole image to the model provider. Do not use it with documents you may not share.
- Scanned PDFs go through OCR in `pymupdf4llm`, which can misplace sections on complex layouts.
- `links` and `summary` are in the ground truth but are not extracted.

**Skills shown in this project:** Python, Pydantic, prompt engineering, LLM API integration, OpenAI SDK, structured JSON output, schema validation, retry and exponential backoff, error handling, OCR, DOCX and PDF parsing, evaluation metrics (precision, recall, F1), telemetry and logging, pytest, ruff, uv, dependency injection.